In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os

import sys
sys.path.append("..")

# Modern import pattern - unified simulation method with strategy pattern
from src import Multicolour_Simulation_Functions
from src.Multicolour_Simulation_Functions import FittingStrategy, SimulationConfig

# Additional required components not integrated into main simulation class
from src import PlottingFunctions
from src import SpectralFunctions
from src import MaskFunctions

# Create main simulation instance (contains IO, PSF, sCMOS, ImageAnalysis dependencies)
MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

# Access integrated components through MSF
IO = MSF.io
I_AF = MSF.image_analysis
sCMOS = MSF.scmos
PSF = MSF.psf

# Create instances of non-integrated components
plotter = PlottingFunctions.Plotter()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()

INFO:Constants:Logging to file: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Code/Python/pyBayerSMLM/logs/Constants_20250902_170808.log


In [2]:
fretfluors = pl.read_csv('/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/FRETFluors/FluorescenceSpectra_FRETfluors_Normalised.csv')

In [3]:
R, G, B, wavelength = S_F.getpixelefficiency()
minwl = np.argmin(np.abs(wavelength - fretfluors['wavelength'].min()))
maxwl = np.argmin(np.abs(wavelength - fretfluors['wavelength'].max()))
wavelength = wavelength[minwl:maxwl+1]
R = R[minwl:maxwl+1]
G = G[minwl:maxwl+1]
B = B[minwl:maxwl+1]
pixel_QYs = np.vstack([B, G, R])

In [4]:
chromophores = fretfluors.columns[1:]

In [5]:
data_folder = '../Camera_Calibrations/Ximea_Camera/'
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))

In [6]:
notch_filter = 'semrock-nf03-405-488-561-635e'
dichroic_mirror = 'semrock-di03-r405-488-561-635-t1-25x36'
shortpass_filter = 'semrock-bsp01-785r'
filters = [notch_filter, dichroic_mirror, shortpass_filter]

In [7]:
n_photon_space = np.logspace(np.log10(500), np.log10(20000), 100)
n_bootstrap = 20000
background_photons = 40
pixel_size = 69
NA = 1.49

In [8]:
import types
smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma" :  1.5}
smoothing_function.extent =  1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [9]:
save_folder = r'/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Data/Simulation/20250902_TestFRETFluors'
if not os.path.isdir(save_folder):
    os.makedirs(save_folder)

In [10]:
image_size = 20
masks = M_F.get_masks(size_x=image_size, size_y=image_size)
camera_parameters = {}
camera_parameters["gain"] = np.full((image_size, image_size), np.median(gain))
camera_parameters["variance"] = np.full((image_size, image_size), np.median(variance))
camera_parameters["readnoise"] = np.full((image_size, image_size), np.median(readnoise))
camera_parameters["offset"] = np.full((image_size, image_size), np.median(offset))
camera_parameters["rqe"] = np.full((image_size, image_size), np.median(rqe))
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ['B', 'G', 'R']
camera_parameters["pixel_order_indices"] = [0, 1, 2]
camera_parameters["masks"] = masks
simulation_config = SimulationConfig(
     n_bootstrap=n_bootstrap,
     background_photons=background_photons,
     NA=NA,
     pixel_size=pixel_size,
    cpu_fraction=0.9,
    save_raw_results=True,
    subtractx0y0=False,
    saverawimages=False,
)

In [ ]:
for dye in chromophores:
        print("Analysing dye {}".format(dye), end="\r",flush=True,)        
        MSF.test_simulation_method(
                    dye='simulated_'+dye,
                    filters=filters,
                    wavelength=wavelength,
                    camera_parameters=camera_parameters,
                    save_folder=save_folder,
                    n_photon_space=n_photon_space,
                    smoothing_function=smoothing_function,
                    strategy=FittingStrategy.STANDARD,  # Explicit strategy specification
                    starting_flag="simulation_",
                    config=simulation_config,  # All additional parameters via config object
                    single_dye_spectrum=fretfluors[dye].to_numpy()
                )

INFO:src.Multicolour_Simulation_Functions:Analysed photon flux 1/100    Time elapsed: 0.184 min


INFO:src.Multicolour_Simulation_Functions:Analysed photon flux 2/100    Time elapsed: 0.369 min


INFO:src.Multicolour_Simulation_Functions:Analysed photon flux 3/100    Time elapsed: 0.526 min


INFO:src.Multicolour_Simulation_Functions:Analysed photon flux 4/100    Time elapsed: 0.748 min


INFO:src.Multicolour_Simulation_Functions:Analysed photon flux 5/100    Time elapsed: 0.903 min


...

INFO:src.Multicolour_Simulation_Functions:Analysed photon flux 81/100    Time elapsed: 19.724 min


INFO:src.Multicolour_Simulation_Functions:Analysed photon flux 82/100    Time elapsed: 19.884 min


INFO:src.Multicolour_Simulation_Functions:Analysed photon flux 83/100    Time elapsed: 20.086 min


INFO:src.Multicolour_Simulation_Functions:Analysed photon flux 84/100    Time elapsed: 20.245 min


INFO:src.Multicolour_Simulation_Functions:Analysed photon flux 85/100    Time elapsed: 20.467 min
